In [7]:
#| default_exp mass.massmesh_module

In [10]:
#| export
import trimesh
import numpy as np
import os
import sys
from trimesh.creation import cylinder
import pandas as pd
from trimesh.visual.color import ColorVisuals
from anthropmass.mass.measurements_heights_module import *

# allow importing your measurement code
#sys.path.append(r"C:\Users\theo4\OneDrive\Skrivbord\BSPLocal copy")
#from measurements_heightsC import get_measurements


ModuleNotFoundError: No module named 'anthropmass'

In [ ]:
#| export
# === HELPERS ===
def create_truncated_cone(radius_top, radius_bottom, height, sections=32):
    angles = np.linspace(0, 2 * np.pi, sections, endpoint=False)
    top = np.stack([radius_top * np.cos(angles),
                    radius_top * np.sin(angles),
                    np.full_like(angles,  height / 2)], axis=1)
    bottom = np.stack([radius_bottom * np.cos(angles),
                       radius_bottom * np.sin(angles),
                       np.full_like(angles, -height / 2)], axis=1)
    vertices = np.vstack([top, bottom])
    faces = []
    for i in range(sections):
        a, b = i, (i + 1) % sections
        c, d = i + sections, (i + 1) % sections + sections
        faces.append([a, b, c])
        faces.append([b, d, c])
    top_center, bottom_center = len(vertices), len(vertices) + 1
    vertices = np.vstack([vertices, [[0, 0, height / 2],
                                     [0, 0, -height / 2]]])
    for i in range(sections):
        next_i = (i + 1) % sections
        faces.append([i, next_i, top_center])
        faces.append([next_i + sections, i + sections, bottom_center])
    return trimesh.Trimesh(vertices=vertices, faces=faces, process=True)

def create_elliptical_frustum_solid(a_top, b_top, a_bot, b_bot, height, n=32):
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False)
    top = np.stack([a_top * np.cos(angles),
                    b_top * np.sin(angles),
                    np.full_like(angles,  height / 2)], axis=1)
    bottom = np.stack([a_bot * np.cos(angles),
                       b_bot * np.sin(angles),
                       np.full_like(angles, -height / 2)], axis=1)
    vertices = np.vstack([top, bottom])
    faces = []
    for i in range(n):
        a, b = i, (i + 1) % n
        c, d = i + n, (i + 1) % n + n
        faces.append([a, b, c])
        faces.append([b, d, c])
    top_center, bottom_center = len(vertices), len(vertices) + 1
    vertices = np.vstack([vertices, [[0, 0, height / 2],
                                     [0, 0, -height / 2]]])
    for i in range(n):
        next_i = (i + 1) % n
        faces.append([i, next_i, top_center])
        faces.append([next_i + n, i + n, bottom_center])
    return trimesh.Trimesh(vertices=vertices, faces=faces, process=True)

def create_elliptical_cylinder(a, b, height, sections=32):
    angles = np.linspace(0, 2 * np.pi, sections, endpoint=False)
    top = np.stack([a * np.cos(angles),
                    b * np.sin(angles),
                    np.full_like(angles,  height / 2)], axis=1)
    bottom = np.stack([a * np.cos(angles),
                       b * np.sin(angles),
                       np.full_like(angles, -height / 2)], axis=1)
    vertices = np.vstack([top, bottom])
    faces = []
    for i in range(sections):
        a_idx, b_idx = i, (i + 1) % sections
        c_idx, d_idx = i + sections, (i + 1) % sections + sections
        faces.append([a_idx, b_idx, c_idx])
        faces.append([b_idx, d_idx, c_idx])
    top_center, bottom_center = len(vertices), len(vertices) + 1
    vertices = np.vstack([vertices, [[0, 0, height / 2],
                                     [0, 0, -height / 2]]])
    for i in range(sections):
        next_i = (i + 1) % sections
        faces.append([i, next_i, top_center])
        faces.append([next_i + sections, i + sections, bottom_center])
    return trimesh.Trimesh(vertices=vertices, faces=faces, process=True)

def mirror_mesh(mesh):
    m = mesh.copy()
    m.apply_scale([-1, 1, 1])
    return m

def rotate_hand_mesh(mesh):
    rot_z = trimesh.transformations.rotation_matrix(
        np.pi/2, [0,0,1], point=[0,0,0])
    mesh.apply_transform(rot_z)
    return mesh

def rotate_foot_mesh(mesh):
    rot_x = trimesh.transformations.rotation_matrix(
        np.pi/2, [-1,0,0], point=[0,0,0])
    rot_z = trimesh.transformations.rotation_matrix(
        np.pi/2, [0,0,1], point=[0,0,0])
    mesh.apply_transform(rot_x)
    mesh.apply_transform(rot_z)
    return mesh

def shift_to_bottom(mesh):
    min_z = mesh.bounds[0][2]
    mesh.apply_translation([0,0,-min_z])
    return mesh

# === MAIN MESH GENERATION ===
def generate_all_meshes(Ansur, inputheight):
    # if Ansur already *is* the measurements table:
    if 'a1' in Ansur.columns:
        measurements = Ansur.iloc[0]
    else:
        measurements = get_measurements(Ansur, inputheight).iloc[0]

    try:
        script_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        # __file__ doesn't exist (e.g. in a notebook), so use cwd
        script_dir = os.getcwd()

    output_dir = os.path.join(script_dir, "model_output")
    os.makedirs(output_dir, exist_ok=True)


    # --- torso ---
    torso_parts = {
        "torso_top": create_elliptical_cylinder(
            measurements["a1"], measurements["b1"], measurements["h1"]
        ),
        "torso_mid": create_elliptical_frustum_solid(
            measurements["a2"], measurements["b2"],
            measurements["a3"], measurements["b3"],
            measurements["h2"]
        ),
        "torso_bottom": create_elliptical_cylinder(
            measurements["a4"], measurements["b4"], measurements["h3"]
        ),
        "neck": cylinder(
            radius=measurements["a9"],
            height=measurements["a8"],
            sections=32
        )
    }

    for name, mesh in torso_parts.items():
        if name == "neck":
            mesh.apply_translation([0, 0, -measurements["a8"]/2])
        else:
            mesh = shift_to_bottom(mesh)
            mesh.apply_transform(
                trimesh.transformations.rotation_matrix(np.pi/2, [0,0,1])
            )
        mesh.fix_normals()
        mesh.export(os.path.join(output_dir, f"{name}.stl"))

    # --- head ---
    head = trimesh.primitives.Sphere(radius=1.0).to_mesh()
    head.apply_transform(
        np.diag([measurements["b8"], measurements["b8"], measurements["a7"], 1])
    )
    head = shift_to_bottom(head)
    head.fix_normals()
    head.export(os.path.join(output_dir, "head.stl"))

    # --- limbs ---
    limbs = {
        "upper_arm": ("trunc_cone", (measurements["r1"], measurements["r2"], measurements["h8"])),
        "forearm": ("trunc_cone", (measurements["r3"], measurements["r4"], measurements["h10"])),
        "upper_leg": ("trunc_cone", (measurements["r1thigh"], measurements["r2thigh"], measurements["h4"])),
        "lower_leg": ("trunc_cone", (measurements["r1shank"], measurements["r2shank"], measurements["h6"])),
        "foot": ("elliptical_frustum", (measurements["b5"], measurements["a5"], measurements["a6"], measurements["b6"], measurements["h7"])),
        "hand": ("ellipsoid", [measurements["r7"], measurements["b7"], measurements["h9"]]),
    }

    for base_name, (shape, dims) in limbs.items():
        for side in ["left","right"]:
            if shape == "ellipsoid":
                m = trimesh.primitives.Sphere(radius=1.0).to_mesh()
                m.apply_transform(np.diag(dims + [1]))
            elif shape == "trunc_cone":
                rt, rb, h = dims
                m = create_truncated_cone(rt, rb, h)
                m.apply_translation([0,0,-h/2])
            else:  # elliptical_frustum
                at, bt, ab, bb, h = dims
                m = create_elliptical_frustum_solid(at, bt, ab, bb, h)

            if base_name == "hand":
                m = rotate_hand_mesh(m)
            elif base_name == "foot":
                m = rotate_foot_mesh(m)

            m = shift_to_bottom(m)

            if side == "right":
                m = mirror_mesh(m)

            if base_name == "foot":
                angle = np.deg2rad(-90)
                rot = trimesh.transformations.rotation_matrix(
                    angle if side=="left" else -angle,
                    [0,0,1]
                )
                m.apply_transform(rot)

            m.fix_normals()
            m.export(os.path.join(output_dir, f"{base_name}_{side}.stl"))

    print(f"✅ Meshes successfully saved to: {output_dir}")





In [ ]:
#| export


# === FUNCTION: Preview Scene ===
def preview_current_colored_geometry(Ansur, inputheight):
    measurements = get_measurements(Ansur, inputheight).iloc[0]

    torso_top_a = measurements["a1"]
    torso_top_b = measurements["b1"]
    torso_capsule_height_top = measurements["h1"]
    torso_mid_top_a = measurements["a2"]
    torso_mid_top_b = measurements["b2"]
    torso_mid_bot_a = measurements["a3"]
    torso_mid_bot_b = measurements["b3"]
    torso_mid_height = measurements["h2"]
    torso_bot_a = measurements["a4"]
    torso_bot_b = measurements["b4"]
    torso_capsule_height_bottom = measurements["h3"]
    neck_radius = measurements["a9"]
    neck_height = measurements["a8"]/5
    
    head_radii = [measurements["b8"], measurements["b8"], measurements["a7"]]
    upper_arm_radius_top = measurements["r1"]
    upper_arm_radius_bottom = measurements["r2"]
    upper_arm_length = measurements["h8"]
    forearm_radius_top = measurements["r3"]
    forearm_radius_bottom = measurements["r4"]
    forearm_length = measurements["h10"]
    upper_leg_radius_top = measurements["r1thigh"]
    upper_leg_radius_bottom = measurements["r2thigh"]
    upper_leg_length = measurements["h4"]
    lower_leg_radius_top = measurements["r1shank"]
    lower_leg_radius_bottom = measurements["r2shank"]
    lower_leg_length = measurements["h6"]
    foot_top_b = measurements["a5"]
    foot_top_a = measurements["b5"]
    foot_bot_a = measurements["a6"]
    foot_bot_b = measurements["b6"]
    foot_height = measurements["h7"]
    hand_radii = [measurements["r7"], measurements["b7"], measurements["h9"]]

    scene = trimesh.Scene()
    x_offset = 0

    torso_parts = {
        "torso_top": (create_elliptical_cylinder(torso_top_a, torso_top_b, torso_capsule_height_top), [200, 200, 200, 255]),
        "torso_mid": (create_elliptical_frustum_solid(torso_mid_top_a, torso_mid_top_b, torso_mid_bot_a, torso_mid_bot_b, torso_mid_height), [150, 150, 150, 255]),
        "torso_bottom": (create_elliptical_cylinder(torso_bot_a, torso_bot_b, torso_capsule_height_bottom), [200, 200, 200, 255]),
        "neck": (cylinder(radius=neck_radius, height=neck_height, sections=32), [255, 255, 100, 255]),
    }

    for name, (mesh, color) in torso_parts.items():
        mesh.visual = ColorVisuals(mesh, face_colors=color)
        if name == "neck":
            mesh.apply_translation([0, 0, -neck_height / 2])
        mesh.apply_translation([x_offset, 0, 0])
        scene.add_geometry(mesh, node_name=name)
        x_offset += 0.6

    limb_parts = {
        "head": ("ellipsoid", head_radii, [255, 255, 255, 255]),
        "upper_arm_left": ("trunc_cone", (upper_arm_radius_top, upper_arm_radius_bottom, upper_arm_length), [255, 0, 0, 255]),
        "forearm_left": ("trunc_cone", (forearm_radius_top, forearm_radius_bottom, forearm_length), [255, 165, 0, 255]),
        "upper_leg_left": ("trunc_cone", (upper_leg_radius_top, upper_leg_radius_bottom, upper_leg_length), [0, 0, 255, 255]),
        "lower_leg_left": ("trunc_cone", (lower_leg_radius_top, lower_leg_radius_bottom, lower_leg_length), [0, 255, 0, 255]),
        "foot_left": ("elliptical_frustum", (foot_top_a, foot_top_b, foot_bot_a, foot_bot_b, foot_height), [160, 82, 45, 255]),
        "hand_left": ("ellipsoid", hand_radii, [255, 192, 203, 255]),
    }

    for part, (shape_type, dims, color) in limb_parts.items():
        if shape_type == "ellipsoid":
            mesh = trimesh.primitives.Sphere(radius=1.0).to_mesh()
            mesh.apply_transform(np.diag(dims + [1.0]))
        elif shape_type == "trunc_cone":
            rt, rb, h = dims
            mesh = create_truncated_cone(rt, rb, h)
            mesh.apply_translation([0, 0, -h / 2])
        elif shape_type == "elliptical_frustum":
            at, bt, ab, bb, h = dims
            mesh = create_elliptical_frustum_solid(at, bt, ab, bb, h)
        else:
            continue
        mesh.visual = ColorVisuals(mesh, face_colors=color)
        mesh.apply_translation([x_offset, 0, 0])
        scene.add_geometry(mesh, node_name=part)
        x_offset += 0.6
        

        # ... your existing mesh creation
        if part == "foot_left":
            angle = np.pi / 2  # 90 degrees
            rotation = trimesh.transformations.rotation_matrix(
                angle=angle,
                direction=[-1, 0, 0],  
                point=[0, 0, 0]
            )
            mesh.apply_transform(rotation)

             # ... your existing mesh creation
        if part == "foot_left":
            angle = np.pi / 2  # 90 degrees
            rotation = trimesh.transformations.rotation_matrix(
                angle=angle,
                direction=[0, 0, 1],  # Z-axis
                point=[0, 0, 0]
            )
            mesh.apply_transform(rotation)


    scene.show(flags={'cull': False})

# === MAIN ===
Ansur = pd.read_csv('../data/processed/ANSURIImalefemale.csv')
subject = Ansur.iloc[[0]]  # Use one subject
generate_all_meshes(subject, 1800)
preview_current_colored_geometry(subject, 1800)





# 🟥 Red = Upper Arm
# 🟧 Orange = Forearm
# 🟦 Blue = Upper Leg
# 🟩 Green = Lower Leg
# ⚪ White = Head
# ⬜ Gray = Torso